In [ ]:
# Core scverse libraries
import scanpy as sc
import anndata as ad

# Data retrieval
import pooch
import polars as pl

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor="white")
sc.settings.verbosity = 0

In [ ]:
!ls

In [ ]:
EXAMPLE_DATA = pooch.create(
    path=pooch.os_cache("scverse_tutorials"),
    base_url="doi:10.6084/m9.figshare.22716739.v1/",
)
EXAMPLE_DATA.load_registry_from_doi()

samples = {
    "s1d1": "s1d1_filtered_feature_bc_matrix.h5",
    "s1d3": "s1d3_filtered_feature_bc_matrix.h5",
}
adatas = {}

for sample_id, filename in samples.items():
    path = EXAMPLE_DATA.fetch(filename)
    sample_adata = sc.read_10x_h5(path)
    sample_adata.var_names_make_unique()
    adatas[sample_id] = sample_adata

adata = ad.concat(adatas, label="sample")

adata.write_h5ad("data/pbmc3k.h5ad")

In [ ]:
adata.raw = adata.copy()

In [ ]:
adata.var

In [ ]:
adata.obs_names_make_unique()

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

In [ ]:
adata.var

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

In [ ]:
adata.obs.head()

In [ ]:
from lets_plot import *

LetsPlot.setup_html()

In [ ]:
# 2. Basic filtering of cells and genes
sc.pp.filter_cells(adata, min_genes=200)  # filter cells with at least 200 genes
sc.pp.filter_genes(adata, min_cells=3)  # filter genes expressed in at least 3 cells

# 5. Normalize total counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# 6. Log-transform the data
sc.pp.log1p(adata)

# 7. Identify highly variable genes (HVGs)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var.highly_variable]

# 8. Scale the data to unit variance and mean zero
sc.pp.scale(adata, max_value=10)

# 9. Perform PCA for dimensionality reduction
sc.tl.pca(adata, n_comps=50)

# 10. Compute the neighborhood graph for clustering
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20)

# 11. Compute UMAP for visualization
sc.tl.umap(adata)
sc.tl.tsne(adata)

# 12. Cluster cells
sc.tl.leiden(adata, resolution=0.5)  # or use sc.tl.louvain

In [ ]:
sc.pl.umap(adata, color="leiden")

In [ ]:
adata.obs.head()

In [ ]:
type(adata.obsm["X_umap"])

In [ ]:
umap = pl.from_numpy(adata.obsm["X_umap"], schema=["umap1", "umap2"])

In [ ]:
umap

In [ ]:
adata.var.head()

In [ ]:
adata.obsm

In [ ]:
adata.var

In [ ]:
adata.var_names

In [ ]:
adata.obs_names

In [ ]:
adata.X.shape  # dimension 0 is cells, dimension 1 is genes

In [ ]:
adata.write("data/pbmc3k_pped.h5ad")